In [1]:
import sys
from pathlib import Path

#Importante para poder acceder al modulo en src
sys.path.append(str(Path("..").resolve()))

In [ ]:
from src.gate_pinn_module import*  
import sympy as sy
import qutip as qt
from itertools import product

Se define una base de kets $\{\ket{0}, \ket{1}\}$ donde:

$$ \ket{0} = \begin{bmatrix}1\\0\end{bmatrix}, \quad \ket{1}= \begin{bmatrix}0 \\ 1\end{bmatrix}$$

In [26]:
#kets en la base computacional
ket_0, ket_1 = basis(2)  

#Definicion de las matrices de pauling
sigma_z=ket_0*ket_0.T - ket_1*ket_1.T
sigma_x = ket_0*ket_1.T + ket_1*ket_0.T
sigma_00=ket_0*ket_0.T
sigma_11=ket_1*ket_1.T #sigma |0><0| o valor esperado de la poblacion |0><0|
sigma_10=ket_1*ket_0.T #sigma |1><0| o valor esperado de la coherencia |0><1|
sigma_m=ket_1*ket_0.T
sigma_p=ket_0*ket_1.T

In [27]:
#Funcion de control a resolver mediante PINN. En éste caso para un hamiltoniano condicionado
control,init_control=control_create(symbols=["Control"],init_control=[0])

#No hay operadores de disipacion "Dinamica cerrada"
l_ops = []

#No hay tasa de disipacion "Dinamica cerrada"
gammas=[]  

In [28]:
pi=sympy.pi
# Estado general en la esfera de Bloch
def state(theta, phi):
    return sympy.cos(theta/2)*ket_0 + sympy.exp(sympy.I*phi)*sympy.sin(theta/2)*ket_1

# Estado después de aplicar la compuerta Hadamard
def h_state(theta, phi):
    return (1/sympy.sqrt(2))*( (sympy.cos(theta/2) + sympy.exp(sympy.I*phi)*sympy.sin(theta/2))*ket_0
                        + (sympy.cos(theta/2) - sympy.exp(sympy.I*phi)*sympy.sin(theta/2))*ket_1 )

In [29]:
#estados adicionales de entrenamiento 

state_1=(state(pi/2,pi/2)*state(pi/2,pi/2).H).evalf() #estado theta=pi/2 y phi=pi/2


h_state_1=(h_state(pi/2,pi/2)*h_state(pi/2,pi/2).H).evalf() #hadamard theta=pi/2 y phi=pi/2

In [ ]:

rho_init = np.array([
    [
        #Estado excitado |0><0|
        [1,0],
        [0,0]
    ],
    [
        #Estado base |1><1|
        [0,0],
        [0,1]
    ],
    [
        #Estado superpuesto
        [1/2,1/2],
        [1/2,1/2]
    ],
    [
        #Estado superpuesto
        [1/2,-1/2],
        [-1/2,1/2]
    ],
        #estado theta=pi/2 y phi=pi/2
        #state_1
])

#Restriccciones para el estado objetivo 
rho_target = np.array([
    [
        #Estado superpuesto
        [1/2,1/2],
        [1/2,1/2]
    ],    
    [
        #Estado superpuesto
        [1/2,-1/2],
        [-1/2,1/2]
    ],
    [
        #Estado excitado |0><0|
        [1,0],
        [0,0]
    ],
    [
        #Estado base |1><1|
        [0,0],
        [0,1]
    ],
        #hadamard theta=pi/2 y phi=pi/2
        #h_state_1 
])

#Constantes
w_0 = 1.0  # Frecuencia de resonancia
h = 1.0   # Constante de Planck reducida

#Definicion del hamiltoniano
hamiltonian = -(h*w_0*sigma_z) + control["Control"]*h*w_0*sigma_x 


In [31]:
#Estados aleatorios para evaluar la fidelidad promedio
estados = 500
qubit = [qt.rand_ket(2) for i in range(estados)] # estados aleatorios para un qubit

# Aplico la hadamard y calculo los estados iniciales en representación de matrices
U=1/np.sqrt(2)*(qt.sigmax()+qt.sigmaz()) #Compuerta Hadamard

U_rho=[]
rho0=[]
for ket in qubit:
    rho=ket*ket.dag()
    rho0.append(rho)
    U_rho.append(U*rho*U.dag())

In [ ]:
#Rango de los parametros de la PINN para el barrido (Solo se barre la tasa de aprendizaje y las neuronas)
#Los demas parametros se definen, con la posibilidad de cambiar el barrido entre parametros.

lr=np.linspace(1e-4,1e-3,10)        #Tasa de aprendizaje
ep=np.arange(5000,15001,2000)       #Epocas de entrenamiento
n=np.arange(100,501,50)             #neuronas
hl=np.arange(1,11,2)                #capas ocultas
chi=np.linspace(1e-5,1e-3,5)        #peso de la regularizacion


In [ ]:
Sweep=[]

#for lr_i, ep_i, n_i, hl_i, chi_i in product(lr, ep, n, hl, chi):
for lr_i, n_i in product(lr,n):
    
    ################################################################################
    #Entrenamiento para la PINN
    ################################################################################   
    
    pinn_properties = {
    "learning_rate": lr_i, #tasa de aprendizaje
    "epochs": 1e4,   #epocas de entrenamiento
    "num_workers":0,
    "batch_size": 500, 
    "neurons": n_i,   #neuronas
    "hidden_layers":6, #capas ocultas
    "time_config": [0.0,10.0, 500], #[t_inicial, t_final, t_size],
    "eta": 0.1, # Peso de la condición del estado objetivo,
    "eta_sc": 1, # Peso de la condición de la traza
    "chi": 0.01e-3, #Hiperparámetro de la regularización l2 ---1e-3---
    "debug_model": False,
    "loss_gate": True
    }   
    #Clase solución
    solution = Solver(hamiltonian, control, init_control, rho_init, rho_target, pinn_properties)
    
    #Entrenamiento de la red
    solution.train_neural_network()
    
    #Vector de tiempo para evaluar el control aprendido
    t_test = torch.linspace(0.0, 10, 500).reshape(-1, 1)
    #t_test.requires_grad=True
    t_net = t_test.detach().numpy()
    control_t= solution.eval_component(t_test, "Control", 0)
    
    ###########################################################################
    #Fidelidad promedio usando qutip 
    ###########################################################################
    
    #Hamiltoniano
    H0=-(h*w_0*qt.sigmaz())
    H1=h*w_0*qt.sigmax()
    Q_H=[H0,[H1,control_t.transpose().squeeze()]]
    
    #Tiempo
    times=np.linspace(0.0,10.0,500)
    
    #Solución de la dinamica cerrada controlada para cada estado aleatorio
    solution_H=[]
    for rho in rho0:
        result= qt.mesolve(Q_H, rho, times, [], [])
        solution_H.append(result.states[-1])
        
    #Fidelidad promedio en T=10 
    fid_H=[]
    for i in range(estados):
        f_H=qt.fidelity(solution_H[i], U_rho[i])
        fid_H.append(f_H**2)
    
    fid_prom_H=sum(fid_H)/estados 
    
    #Agregamos los parametros, control y fidelidad a un diccionario
    Sweep.append({
        "learning rate": lr_i,
        "epochs": 1e4, #ep_i,
        "neurons": n_i,
        "hidden_layers": 6, #hl_i,
        "chi": 0.01e-3,#chi_i,
        "A. fidelity": fid_prom_H,
        "control": control_t,
        "loss": solution.loss_history,
        "L_model": solution.loss_model,
        "L_control": solution.loss_control,
        "L_const": solution.loss_const,
        "L_reg" : solution.loss_reg
    })

#Almacenamos el barrido en un archivo binario 
#(Si se cambian los parametros del barrido, ¡cambiar el nombre de estos en el "filename"!)
#Ejemplo: (Epocas y capas ocultas), entonces: f"../Results/HADAMARD_ep{ep[0]:.0e}-{ep[-1]:.0e}_hl{hl[0]}-{hl[-1]}.npy"

filename = f"../Results/HADAMARD_lr{lr[0]:.0e}-{lr[-1]:.0e}_neurons{n[0]}-{n[-1]}.npy"
np.save(filename, Sweep, allow_pickle=True)